# PS S6E6 - 012 Bias search on 010 shared002 LGB

Experiment: `exp_20260605_012_lgb010_bias_search`

This notebook is derived from the 007 bias-search notebook, but targets 010 shared002 LGB as-is.

Purpose:
- Load 010 OOF/test probabilities.
- Search class-wise bias on OOF to maximize balanced accuracy.
- Apply the best bias to OOF/test probabilities.
- Save blend-ready adjusted OOF/test `.npy` files with the full `exp_*` id in the filename.

Base:
- `exp_20260604_010_shared002_lgb_as_is`
- CV: `0.9625129421154345`
- Public LB: `0.96357`

In [1]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260605_012_lgb010_bias_search"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

REF_010_EXP_ID = "exp_20260604_010_shared002_lgb_as_is"
REF_010_CV = 0.9625129421154345
REF_010_PUBLIC_LB = 0.96357

OOF_010_PATH = "/kaggle/working/exp_20260604_010_shared002_lgb_as_is/oof_exp_20260604_010_shared002_lgb_as_is_proba.npy"
PRED_010_PATH = "/kaggle/working/exp_20260604_010_shared002_lgb_as_is/pred_exp_20260604_010_shared002_lgb_as_is_proba.npy"

OOF_BIASED_NPY = OUTDIR / "oof_exp_20260605_012_lgb010_bias_search_proba.npy"
PRED_BIASED_NPY = OUTDIR / "pred_exp_20260605_012_lgb010_bias_search_proba.npy"

print("OUTDIR:", OUTDIR)
print("OOF_BIASED_NPY:", OOF_BIASED_NPY)
print("PRED_BIASED_NPY:", PRED_BIASED_NPY)

OUTDIR: /kaggle/working/exp_20260605_012_lgb010_bias_search
OOF_BIASED_NPY: /kaggle/working/exp_20260605_012_lgb010_bias_search/oof_exp_20260605_012_lgb010_bias_search_proba.npy
PRED_BIASED_NPY: /kaggle/working/exp_20260605_012_lgb010_bias_search/pred_exp_20260605_012_lgb010_bias_search_proba.npy


In [2]:
def save_json(obj, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def resolve_path(path_like: str) -> Path:
    p = Path(path_like)
    if p.exists():
        return p

    fname = p.name
    matches = []
    for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
        if root.exists():
            matches.extend(root.rglob(fname))

    matches = sorted(set(matches))
    if len(matches) == 1:
        print(f"[resolve_path] {path_like} -> {matches[0]}")
        return matches[0]
    if len(matches) > 1:
        print(f"[resolve_path] Multiple candidates for {fname}:")
        for m in matches[:30]:
            print(" -", m)
        print("[resolve_path] Using first candidate:", matches[0])
        return matches[0]

    raise FileNotFoundError(
        f"Could not resolve path: {path_like}\n"
        f"Expected filename: {fname}\n"
        "Add 010 outputs as Kaggle input dataset or edit OOF_010_PATH/PRED_010_PATH."
    )

def proba_to_logp(proba: np.ndarray, eps: float = 1e-15) -> np.ndarray:
    return np.log(np.clip(proba, eps, 1.0))

def softmax_np(x: np.ndarray, axis: int = 1) -> np.ndarray:
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def apply_bias_to_proba(proba: np.ndarray, bias: np.ndarray) -> np.ndarray:
    logp = proba_to_logp(proba)
    return softmax_np(logp + bias.reshape(1, -1), axis=1)

def predict_with_bias(logp: np.ndarray, bias: np.ndarray) -> np.ndarray:
    return np.argmax(logp + bias.reshape(1, -1), axis=1)

def class_recalls(y_true: np.ndarray, y_pred: np.ndarray, class_names: list[str]) -> dict:
    out = {}
    for i, cls in enumerate(class_names):
        mask = y_true == i
        out[f"recall_{cls}"] = float((y_pred[mask] == i).mean()) if mask.any() else float("nan")
    return out

def eval_pred(y_true: np.ndarray, y_pred: np.ndarray, class_names: list[str]) -> dict:
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred))}
    res.update(class_recalls(y_true, y_pred, class_names))
    return res

def validate_proba(name: str, arr: np.ndarray, n_rows: int, n_classes: int) -> None:
    assert arr.shape == (n_rows, n_classes), (name, arr.shape, n_rows, n_classes)
    assert np.isfinite(arr).all(), name
    row_sum = arr.sum(axis=1)
    assert np.allclose(row_sum, 1.0, atol=1e-4), (name, row_sum.min(), row_sum.max())
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

In [3]:
for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)

assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof_path = resolve_path(OOF_010_PATH)
pred_path = resolve_path(PRED_010_PATH)

oof_proba_raw = np.load(oof_path).astype(np.float32)
test_proba_raw = np.load(pred_path).astype(np.float32)

validate_proba("oof_proba_raw", oof_proba_raw, len(train), n_classes)
validate_proba("test_proba_raw", test_proba_raw, len(test), n_classes)

oof_logp = proba_to_logp(oof_proba_raw)
test_logp = proba_to_logp(test_proba_raw)

print("train:", train.shape)
print("test:", test.shape)
print("class_names:", class_names)
print("oof_path:", oof_path)
print("pred_path:", pred_path)
print("oof_proba_raw:", oof_proba_raw.shape)
print("test_proba_raw:", test_proba_raw.shape)

[resolve_path] /kaggle/working/exp_20260604_010_shared002_lgb_as_is/oof_exp_20260604_010_shared002_lgb_as_is_proba.npy -> /kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260604_010_shared002_lgb_as_is_proba.npy
[resolve_path] /kaggle/working/exp_20260604_010_shared002_lgb_as_is/pred_exp_20260604_010_shared002_lgb_as_is_proba.npy -> /kaggle/input/datasets/mizushimatoshihiko/npy-files/pred_exp_20260604_010_shared002_lgb_as_is_proba.npy
train: (577347, 12)
test: (247435, 11)
class_names: ['GALAXY', 'QSO', 'STAR']
oof_path: /kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260604_010_shared002_lgb_as_is_proba.npy
pred_path: /kaggle/input/datasets/mizushimatoshihiko/npy-files/pred_exp_20260604_010_shared002_lgb_as_is_proba.npy
oof_proba_raw: (577347, 3)
test_proba_raw: (247435, 3)


In [4]:
raw_bias = np.zeros(n_classes, dtype=np.float64)
raw_pred = predict_with_bias(oof_logp, raw_bias)
raw_eval = eval_pred(y, raw_pred, class_names)

print("Raw 010:")
print(json.dumps(raw_eval, indent=2, ensure_ascii=False))
print("delta vs REF_010_CV:", raw_eval["balanced_accuracy"] - REF_010_CV)
print(confusion_matrix(y, raw_pred))
print(classification_report(y, raw_pred, target_names=class_names))

Raw 010:
{
  "balanced_accuracy": 0.9625129421154345,
  "recall_GALAXY": 0.9676751086150259,
  "recall_QSO": 0.9674073568202965,
  "recall_STAR": 0.9524563609109811
}
delta vs REF_010_CV: 0.0
[[365278   4472   7730]
 [  2451 113325   1367]
 [  3529    404  78791]]
              precision    recall  f1-score   support

      GALAXY       0.98      0.97      0.98    377480
         QSO       0.96      0.97      0.96    117143
        STAR       0.90      0.95      0.92     82724

    accuracy                           0.97    577347
   macro avg       0.95      0.96      0.95    577347
weighted avg       0.97      0.97      0.97    577347



In [5]:
def grid_search_bias_2d(logp, y_true, qso_values, star_values, stage_name):
    rows = []
    best_score = -1.0
    best_bias = None

    for bq in qso_values:
        for bs in star_values:
            bias = np.array([0.0, float(bq), float(bs)], dtype=np.float64)
            pred = predict_with_bias(logp, bias)
            score = float(balanced_accuracy_score(y_true, pred))

            rows.append({
                "stage": stage_name,
                "bias_GALAXY": 0.0,
                "bias_QSO": float(bq),
                "bias_STAR": float(bs),
                "balanced_accuracy": score,
            })

            if score > best_score:
                best_score = score
                best_bias = bias.copy()

        print(f"{stage_name}: bq={bq:.6f}, best={best_score:.9f}, bias={best_bias}")

    return {"score": best_score, "bias": best_bias}, pd.DataFrame(rows)

# Wider than 007 because 010 has a different recall profile.
coarse_qso = np.round(np.arange(-0.200, 0.200 + 1e-12, 0.010), 6)
coarse_star = np.round(np.arange(-0.200, 0.200 + 1e-12, 0.010), 6)
best_coarse, coarse_df = grid_search_bias_2d(oof_logp, y, coarse_qso, coarse_star, "coarse")

cq, cs = best_coarse["bias"][1], best_coarse["bias"][2]
local_qso = np.round(np.arange(cq - 0.020, cq + 0.020 + 1e-12, 0.002), 6)
local_star = np.round(np.arange(cs - 0.020, cs + 0.020 + 1e-12, 0.002), 6)
best_local, local_df = grid_search_bias_2d(oof_logp, y, local_qso, local_star, "local")

uq, us = best_local["bias"][1], best_local["bias"][2]
ultra_qso = np.round(np.arange(uq - 0.004, uq + 0.004 + 1e-12, 0.0005), 6)
ultra_star = np.round(np.arange(us - 0.004, us + 0.004 + 1e-12, 0.0005), 6)
best_ultra, ultra_df = grid_search_bias_2d(oof_logp, y, ultra_qso, ultra_star, "ultra_local")

search_df = pd.concat([coarse_df, local_df, ultra_df], ignore_index=True)
search_df = search_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)

best_bias = best_ultra["bias"]
best_score = best_ultra["score"]

display(search_df.head(30))
print("raw_score:", raw_eval["balanced_accuracy"])
print("best_score:", best_score)
print("delta_vs_raw:", best_score - raw_eval["balanced_accuracy"])
print("best_bias:", best_bias)

coarse: bq=-0.200000, best=0.962624864, bias=[ 0.  -0.2  0.2]
coarse: bq=-0.190000, best=0.962636044, bias=[ 0.   -0.19  0.2 ]
coarse: bq=-0.180000, best=0.962646249, bias=[ 0.   -0.18  0.2 ]
coarse: bq=-0.170000, best=0.962652817, bias=[ 0.   -0.17  0.2 ]
coarse: bq=-0.160000, best=0.962656324, bias=[ 0.   -0.16  0.2 ]
coarse: bq=-0.150000, best=0.962656324, bias=[ 0.   -0.16  0.2 ]
coarse: bq=-0.140000, best=0.962660676, bias=[ 0.   -0.14  0.2 ]
coarse: bq=-0.130000, best=0.962670476, bias=[ 0.   -0.13  0.2 ]
coarse: bq=-0.120000, best=0.962693235, bias=[ 0.   -0.12  0.2 ]
coarse: bq=-0.110000, best=0.962704905, bias=[ 0.   -0.11  0.2 ]
coarse: bq=-0.100000, best=0.962756119, bias=[ 0.  -0.1  0.2]
coarse: bq=-0.090000, best=0.962774455, bias=[ 0.   -0.09  0.2 ]
coarse: bq=-0.080000, best=0.962825963, bias=[ 0.   -0.08  0.2 ]
coarse: bq=-0.070000, best=0.962840662, bias=[ 0.   -0.07  0.2 ]
coarse: bq=-0.060000, best=0.962847231, bias=[ 0.   -0.06  0.2 ]
coarse: bq=-0.050000, best=0.96

,stage,bias_GALAXY,bias_QSO,bias_STAR,balanced_accuracy
0,ultra_local,0.0,0.2145,0.2230,0.963228
1,ultra_local,0.0,0.2140,0.2230,0.963228
2,ultra_local,0.0,0.2150,0.2230,0.963227
3,ultra_local,0.0,0.2140,0.2215,0.963227
4,ultra_local,0.0,0.2145,0.2215,0.963227
5,ultra_local,0.0,0.2155,0.2230,0.963227
6,ultra_local,0.0,0.2135,0.2230,0.963226
7,ultra_local,0.0,0.2145,0.2220,0.963226
8,ultra_local,0.0,0.2140,0.2220,0.963226
9,ultra_local,0.0,0.2140,0.2225,0.963226


raw_score: 0.9625129421154345
best_score: 0.9632282943283176
delta_vs_raw: 0.0007153522128831025
best_bias: [0.    0.214 0.223]


In [6]:
oof_proba_biased = apply_bias_to_proba(oof_proba_raw, best_bias).astype(np.float32)
test_proba_biased = apply_bias_to_proba(test_proba_raw, best_bias).astype(np.float32)

validate_proba("oof_proba_biased", oof_proba_biased, len(train), n_classes)
validate_proba("test_proba_biased", test_proba_biased, len(test), n_classes)

biased_oof_pred = oof_proba_biased.argmax(axis=1)
biased_eval = eval_pred(y, biased_oof_pred, class_names)

print("Biased 012:")
print(json.dumps(biased_eval, indent=2, ensure_ascii=False))
print("delta vs raw:", biased_eval["balanced_accuracy"] - raw_eval["balanced_accuracy"])
print("delta vs REF_010_CV:", biased_eval["balanced_accuracy"] - REF_010_CV)
print(confusion_matrix(y, biased_oof_pred))
print(classification_report(y, biased_oof_pred, target_names=class_names))

compare_eval = pd.DataFrame([
    {"version": "raw_010", **raw_eval},
    {"version": "biased_012", **biased_eval},
])
display(compare_eval)

print("OOF proba raw/biased diff abs mean:", float(np.mean(np.abs(oof_proba_biased - oof_proba_raw))))
print("TEST proba raw/biased diff abs mean:", float(np.mean(np.abs(test_proba_biased - test_proba_raw))))

Biased 012:
{
  "balanced_accuracy": 0.9632282943283176,
  "recall_GALAXY": 0.9646338878881,
  "recall_QSO": 0.9689439403122679,
  "recall_STAR": 0.9561070547845849
}
delta vs raw: 0.0007153522128831025
delta vs REF_010_CV: 0.0007153522128831025
[[364130   4817   8533]
 [  2255 113505   1383]
 [  3229    402  79093]]
              precision    recall  f1-score   support

      GALAXY       0.99      0.96      0.97    377480
         QSO       0.96      0.97      0.96    117143
        STAR       0.89      0.96      0.92     82724

    accuracy                           0.96    577347
   macro avg       0.94      0.96      0.95    577347
weighted avg       0.97      0.96      0.96    577347



,version,balanced_accuracy,recall_GALAXY,recall_QSO,recall_STAR
0,raw_010,0.962513,0.967675,0.967407,0.952456
1,biased_012,0.963228,0.964634,0.968944,0.956107


OOF proba raw/biased diff abs mean: 0.00261222873814404
TEST proba raw/biased diff abs mean: 0.0027409368194639683


In [7]:
biased_test_pred = test_proba_biased.argmax(axis=1)
biased_test_labels = le.inverse_transform(biased_test_pred)

submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET_COL: biased_test_labels,
})

assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

print(submission[TARGET_COL].value_counts(normalize=True))
display(submission.head())

submission_path = OUTDIR / "submission_lgb010_bias_search.csv"
submission.to_csv(submission_path, index=False)
print("saved:", submission_path)

class
GALAXY    0.640791
QSO       0.205391
STAR      0.153818
Name: proportion, dtype: float64


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


saved: /kaggle/working/exp_20260605_012_lgb010_bias_search/submission_lgb010_bias_search.csv


In [8]:
np.save(OUTDIR / "bias_vector.npy", best_bias)

# Blend-ready .npy files with exp_* in filenames.
np.save(OOF_BIASED_NPY, oof_proba_biased)
np.save(PRED_BIASED_NPY, test_proba_biased)

# Optional raw copies with explicit exp name for audit.
np.save(OUTDIR / "oof_exp_20260605_012_lgb010_bias_search_raw010_proba.npy", oof_proba_raw)
np.save(OUTDIR / "pred_exp_20260605_012_lgb010_bias_search_raw010_proba.npy", test_proba_raw)

oof_df = pd.DataFrame({
    ID_COL: train[ID_COL].values,
    "y_true": train[TARGET_COL].astype(str).values,
    "raw_pred": le.inverse_transform(raw_pred),
    "biased_pred": le.inverse_transform(biased_oof_pred),
})
for i, cls in enumerate(class_names):
    oof_df[f"raw_proba_{cls}"] = oof_proba_raw[:, i]
    oof_df[f"biased_proba_{cls}"] = oof_proba_biased[:, i]
    oof_df[f"logp_{cls}"] = oof_logp[:, i]
oof_df.to_csv(OUTDIR / "oof_lgb010_bias_search.csv", index=False)

pred_df = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    "biased_pred_label": biased_test_labels,
})
for i, cls in enumerate(class_names):
    pred_df[f"raw_proba_{cls}"] = test_proba_raw[:, i]
    pred_df[f"biased_proba_{cls}"] = test_proba_biased[:, i]
    pred_df[f"logp_{cls}"] = test_logp[:, i]
pred_df.to_csv(OUTDIR / "pred_lgb010_bias_search.csv", index=False)

search_df.to_csv(OUTDIR / "bias_search_results.csv", index=False)
compare_eval.to_csv(OUTDIR / "raw_vs_biased_oof_eval.csv", index=False)

cv_result = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "model": "LightGBM 010 postprocess",
    "status": "completed",
    "metric": "balanced_accuracy_score",
    "reference_exp_id": REF_010_EXP_ID,
    "raw_cv_score": float(raw_eval["balanced_accuracy"]),
    "biased_cv_score": float(biased_eval["balanced_accuracy"]),
    "delta_biased_minus_raw": float(biased_eval["balanced_accuracy"] - raw_eval["balanced_accuracy"]),
    "reference_010_public_lb": REF_010_PUBLIC_LB,
    "class_names": class_names,
    "label_mapping": {str(cls): int(i) for i, cls in enumerate(class_names)},
    "bias": {
        "GALAXY": float(best_bias[0]),
        "QSO": float(best_bias[1]),
        "STAR": float(best_bias[2]),
    },
    "raw_eval": raw_eval,
    "biased_eval": biased_eval,
    "use_original": False,
    "use_fe": True,
    "base_fe_family": "shared002_lgb_fe_as_is",
    "use_bias_search": True,
    "adjusted_probability_method": "softmax(log(raw_proba) + class_bias)",
    "search_ranges": {
        "coarse": {"min": -0.20, "max": 0.20, "step": 0.01},
        "local": {"around_best": 0.02, "step": 0.002},
        "ultra_local": {"around_best": 0.004, "step": 0.0005},
    },
    "submission_path": str(submission_path),
    "oof_path": str(OOF_BIASED_NPY),
    "pred_path": str(PRED_BIASED_NPY),
    "raw_010_oof_path": str(oof_path),
    "raw_010_pred_path": str(pred_path),
}
save_json(cv_result, OUTDIR / "cv_result.json")

print("Artifacts saved to:", OUTDIR)
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Artifacts saved to: /kaggle/working/exp_20260605_012_lgb010_bias_search
 - bias_search_results.csv
 - bias_vector.npy
 - cv_result.json
 - oof_exp_20260605_012_lgb010_bias_search_proba.npy
 - oof_exp_20260605_012_lgb010_bias_search_raw010_proba.npy
 - oof_lgb010_bias_search.csv
 - pred_exp_20260605_012_lgb010_bias_search_proba.npy
 - pred_exp_20260605_012_lgb010_bias_search_raw010_proba.npy
 - pred_lgb010_bias_search.csv
 - raw_vs_biased_oof_eval.csv
 - submission_lgb010_bias_search.csv


In [9]:
registry_path = WORKDIR / "oof_registry.csv"

registry_row = {
    "exp_id": EXP_ID,
    "model_family": "LightGBM",
    "feature_family": "shared002_lgb_fe_as_is",
    "cv_metric": "balanced_accuracy_score",
    "cv_score": float(biased_eval["balanced_accuracy"]),
    "raw_cv_score": float(raw_eval["balanced_accuracy"]),
    "cv_delta_vs_raw": float(biased_eval["balanced_accuracy"] - raw_eval["balanced_accuracy"]),
    "public_lb": np.nan,
    "use_original": False,
    "use_fe": True,
    "use_bias_search": True,
    "oof_path": str(OOF_BIASED_NPY),
    "pred_path": str(PRED_BIASED_NPY),
    "submission_path": str(submission_path),
    "role": "postprocess_bias_candidate",
    "status": "completed",
    "keep_hold_drop": "pending_public_lb",
    "notes": "Class-wise bias search on 010 shared002 LGB as-is. Blend-ready adjusted OOF/pred proba saved with exp_* filenames.",
}

if registry_path.exists():
    registry = pd.read_csv(registry_path)
    registry = registry[registry["exp_id"] != EXP_ID]
    registry = pd.concat([registry, pd.DataFrame([registry_row])], ignore_index=True)
else:
    registry = pd.DataFrame([registry_row])

registry.to_csv(registry_path, index=False)
registry.to_csv(OUTDIR / "oof_registry.csv", index=False)

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Bias search on 010 shared002 LightGBM as-is",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "model": "LightGBM 010 postprocess",
        "created_at": "2026-06-05",
    },
    "objective": {
        "summary": (
            "Apply class-wise bias search to 010 shared002 LGB as-is OOF/test probabilities. "
            "Do not retrain. GALAXY bias is fixed to 0 and QSO/STAR biases are searched. "
            "Save adjusted OOF/test probability matrices so 012 can be added to future blend notebooks."
        ),
        "success_criteria": [
            "load 010 OOF and test predictions",
            "confirm raw 010 OOF score",
            "search class-wise bias on OOF",
            "save adjusted OOF proba as npy",
            "save adjusted test pred proba as npy",
            "save biased submission",
            "save bias vector and search results",
            "update oof_registry",
        ],
    },
    "data": {
        "train_path": str(TRAIN_PATH),
        "test_path": str(TEST_PATH),
        "sample_submission_path": str(SAMPLE_SUB_PATH),
        "target_col": TARGET_COL,
        "id_col": ID_COL,
        "raw_010_oof_path": str(oof_path),
        "raw_010_pred_path": str(pred_path),
        "use_original": False,
    },
    "bias_search": {
        "bias_space": {
            "GALAXY": "fixed_0",
            "QSO": "searched",
            "STAR": "searched",
        },
        "best_bias": {
            "GALAXY": float(best_bias[0]),
            "QSO": float(best_bias[1]),
            "STAR": float(best_bias[2]),
        },
        "raw_cv": float(raw_eval["balanced_accuracy"]),
        "biased_cv": float(biased_eval["balanced_accuracy"]),
        "delta_biased_minus_raw": float(biased_eval["balanced_accuracy"] - raw_eval["balanced_accuracy"]),
        "raw_eval": raw_eval,
        "biased_eval": biased_eval,
        "adjusted_probability_method": "softmax(log(raw_proba) + class_bias)",
    },
    "outputs": {
        "submission": "submission_lgb010_bias_search.csv",
        "bias_vector": "bias_vector.npy",
        "oof_proba": OOF_BIASED_NPY.name,
        "pred_proba": PRED_BIASED_NPY.name,
        "bias_search_results": "bias_search_results.csv",
        "raw_vs_biased_oof_eval": "raw_vs_biased_oof_eval.csv",
        "oof_csv": "oof_lgb010_bias_search.csv",
        "pred_csv": "pred_lgb010_bias_search.csv",
        "cv_result": "cv_result.json",
        "registry": "oof_registry.csv",
    },
    "judgement": {
        "status": "pending_public_lb",
        "reason": (
            "Bias search can improve OOF balanced accuracy, but may overfit class thresholds. "
            "Adoption depends on Public LB and future blend behavior against 011 best blend."
        ),
        "next_action": [
            "Submit submission_lgb010_bias_search.csv",
            "Record Public LB",
            "Add oof_exp_20260605_012_lgb010_bias_search_proba.npy and pred_exp_20260605_012_lgb010_bias_search_proba.npy to npy-files dataset",
            "Compare 012 against raw 010 and 011 best blend",
            "If useful, include 012 in next blend notebook",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("registry saved:", registry_path)
print("memo saved:", memo_path)
display(registry.tail())

registry saved: /kaggle/working/oof_registry.csv
memo saved: /kaggle/working/exp_20260605_012_lgb010_bias_search/memo.yml


,exp_id,model_family,feature_family,cv_metric,cv_score,raw_cv_score,cv_delta_vs_raw,public_lb,use_original,use_fe,use_bias_search,oof_path,pred_path,submission_path,role,status,keep_hold_drop,notes
0,exp_20260605_012_lgb010_bias_search,LightGBM,shared002_lgb_fe_as_is,balanced_accuracy_score,0.963228,0.962513,0.000715,NaN,False,True,True,/kaggle/working/exp_20260605_012_lgb010_bias_s...,/kaggle/working/exp_20260605_012_lgb010_bias_s...,/kaggle/working/exp_20260605_012_lgb010_bias_s...,postprocess_bias_candidate,completed,pending_public_lb,Class-wise bias search on 010 shared002 LGB as...
